# Model IO之提示词的使用
整体要讲的内容：

1、PromptTemplate

2、ChatPromptTemplate


3、少量示例的提示词模板：

FewShotPromptTemplate

FewShotChatMessagePromptTemplate

4、了解：PipelinePromptTemplate、自定义提示词模板

5、如何从物理文件中读取提示词

## 1、PromptTemplate的使用

### ① 两种实例化的方式

第1种：构造方法

In [1]:

from langchain_core.prompts import PromptTemplate

template_text = "你是一个人工智能助手，名字叫{name}"

# 至少要传入两个参数：template、input_variables
prompt_template = PromptTemplate(
    template=template_text,
    input_variables=["name"],
)

# print(prompt_template)

# 调用
str = prompt_template.format(name="小智")
print(str)
print(type(str))

你是一个人工智能助手，名字叫小智
<class 'str'>


In [2]:
from langchain.prompts import PromptTemplate

# 定义模板：描述主题的应用
template = PromptTemplate(template="请简要描述{topic}的应用。",
                          input_variables=["topic"])

print(template)

# 使用模板生成提示词
prompt_1 = template.format(topic="机器学习")
prompt_2 = template.format(topic="自然语言处理")

print("提示词1:", prompt_1)
print("提示词2:", prompt_2)

input_variables=['topic'] input_types={} partial_variables={} template='请简要描述{topic}的应用。'
提示词1: 请简要描述机器学习的应用。
提示词2: 请简要描述自然语言处理的应用。


In [3]:
from langchain.prompts import PromptTemplate

#定义多变量模板
template = PromptTemplate(
    template="请评价{product}的优缺点，包括{aspect1}和{aspect2}。",
    input_variables=["product", "aspect1", "aspect2"])

#使用模板生成提示词
prompt_1 = template.format(product="智能手机", aspect1="电池续航", aspect2="拍照质量")

print("提示词1:",prompt_1)

提示词1: 请评价智能手机的优缺点，包括电池续航和拍照质量。


In [4]:
from langchain.prompts import PromptTemplate

#定义多变量模板。
template = PromptTemplate(
    template="请评价{product}的优缺点，包括{aspect1}和{aspect2}。",
    input_variables=["product", "aspect1", "aspect2"])

#使用模板生成提示词
# 说明：在调用format()函数时，一定要将所有的参数都进行赋值，才可以得到正确的字符串返回结果。否则报错
prompt_1 = template.format(product="智能手机", aspect1="电池续航")
# prompt_1 = template.format(product="智能手机", aspect1="电池续航", aspect2="拍照质量")

print("提示词1:",prompt_1)

KeyError: 'aspect2'

第2种：from_template()   ---推荐的写法。方法内只需要传入template参数即可。


In [5]:
template = PromptTemplate.from_template(
    template="请评价{product}的优缺点，包括{aspect1}和{aspect2}。")

prompt_1 = template.format(product="智能手机", aspect1="电池续航", aspect2="拍照质量")

print("提示词1:",prompt_1)

提示词1: 请评价智能手机的优缺点，包括电池续航和拍照质量。


In [6]:
#1.导入相关的包
from langchain_core.prompts import PromptTemplate

# 2.定义提示词模版对象
text = """
Tell me a joke
"""

prompt_template = PromptTemplate.from_template(template = text)
# 3.默认使用f-string进行格式化（返回格式好的字符串）
prompt = prompt_template.format()
print(prompt)


Tell me a joke



### ② 两种特殊的结构：部分提示词模板（掌握）；组合提示词的使用（了解）

① 部分提示词模板的使用

In [7]:
template = PromptTemplate.from_template(
    template="请评价{product}的优缺点，包括{aspect1}和{aspect2}。",
    partial_variables={"aspect2":"拍照质量"}
)

result = template.format(product="智能手机", aspect1="电池续航",)
# result = template.format(product="智能手机", aspect1="电池续航",aspect2="拍照质量111")
print(result)

请评价智能手机的优缺点，包括电池续航和拍照质量。


In [8]:
template = PromptTemplate.from_template(
    template="请评价{product}的优缺点，包括{aspect1}和{aspect2}。",
    partial_variables={"aspect2":"拍照质量","aspect1":"电池续航"}
)

result = template.format(product="智能手机", )
print(result)

请评价智能手机的优缺点，包括电池续航和拍照质量。


In [9]:
template = PromptTemplate(
    template="请评价{product}的优缺点，包括{aspect1}和{aspect2}。",
    input_variables=["product", "aspect1","aspect2"],
    partial_variables={"aspect2":"拍照质量","aspect1":"电池续航"}
)

str = template.format(product = "笔记本电脑")
print(str)

请评价笔记本电脑的优缺点，包括电池续航和拍照质量。


In [10]:
template = PromptTemplate(
    template="请评价{product}的优缺点，包括{aspect1}和{aspect2}。",
    input_variables=["product", "aspect1","aspect2"],
    #partial_variables={"aspect2":"拍照质量","aspect1":"电池续航"}
)
#调用完partial()方法以后，返回值是一个提示词模板对象，其内部的参数只有product。
# 而template提示词模板对象本身仍然有3个参数
template1 = template.partial(aspect1="电池续航",aspect2="拍照质量")

str = template1.format(product = "笔记本电脑")
print(str)

请评价笔记本电脑的优缺点，包括电池续航和拍照质量。


In [11]:
template = PromptTemplate(
    template="请评价{product}的优缺点，包括{aspect1}和{aspect2}。",
    input_variables=["product", "aspect1","aspect2"],
).partial(aspect1="电池续航",aspect2="拍照质量")

str = template.format(product = "笔记本电脑")
print(str)

请评价笔记本电脑的优缺点，包括电池续航和拍照质量。


In [ ]:
template = PromptTemplate.from_template(  # 使用from_template方法创建提示词模板
    template="请评价{product}的优缺点，包括{aspect1}和{aspect2}。",  # 定义提示词模板内容，包含三个变量占位符
    # partial_variables={"aspect2":"拍照质量"}  # 这行被注释掉了，原本用于设置部分变量的默认值
)

template1 = template.partial(aspect1="电池续航",aspect2="拍照质量")  # 使用partial方法预设部分变量的值，返回新的模板对象

result = template1.format(product="智能手机",)  # 使用format方法填充剩余变量，生成最终提示词
# result = template.format(product="智能手机", aspect1="电池续航",aspect2="拍照质量111")  # 这是另一种直接填充所有变量的方式（被注释）
print(result)  # 打印生成的提示词结果

请评价智能手机的优缺点，包括电池续航和拍照质量。


② 组合提示词的使用


In [13]:
from langchain_core.prompts import PromptTemplate

template = (
    PromptTemplate.from_template(template = "Tell me a joke about {topic}")
    + ", make it funny"
    + "\n\nand in {language}"
)

prompt = template.format(topic="sports", language="spanish")
print(prompt)

Tell me a joke about sports, make it funny

and in spanish


### ③ 模板的调用：format() / invoke(

format(): 形参：是一个一个的参数；返回值是字符串类型

invoke(): 形参：是字典类型；  返回值是PromptValue   ---推荐

In [14]:
template = PromptTemplate.from_template(
    template="请评价{product}的优缺点，包括{aspect1}和{aspect2}。")

prompt_1 = template.format(product="智能手机", aspect1="电池续航", aspect2="拍照质量")

print(prompt_1)
print(type(prompt_1))

请评价智能手机的优缺点，包括电池续航和拍照质量。
<class 'str'>


In [15]:
template = PromptTemplate.from_template(
    template="请评价{product}的优缺点，包括{aspect1}和{aspect2}。")

# prompt_1 = template.format(product="智能手机", aspect1="电池续航", aspect2="拍照质量")
prompt_1 = template.invoke({"product":"智能手机","aspect1":"电池续航","aspect2":"拍照质量"})

print(prompt_1)
print(type(prompt_1))

text='请评价智能手机的优缺点，包括电池续航和拍照质量。'
<class 'langchain_core.prompt_values.StringPromptValue'>


In [16]:
from langchain_core.prompts import PromptTemplate

prompt_template = (
        PromptTemplate.from_template("Tell me a joke about {topic}")
        + ", make it funny"
        + " and in {language}"
)

prompt = prompt_template.invoke({"topic":"sports", "language":"spanish"})
print(prompt)
print(type(prompt))
print(prompt.text)
print(type(prompt.text))


print(prompt.to_string()) #将PromptValue类型转换为Str类型

text='Tell me a joke about sports, make it funny and in spanish'
<class 'langchain_core.prompt_values.StringPromptValue'>
Tell me a joke about sports, make it funny and in spanish
<class 'str'>
Tell me a joke about sports, make it funny and in spanish


### ④ 结合大模型的调用

使用非对话大模型进行调用：

In [17]:
import os
import dotenv
from langchain_openai import OpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 非对话大模型
llm = OpenAI()

In [21]:
template = PromptTemplate.from_template(
    template="请评价{product}的优缺点，包括{aspect1}和{aspect2}。")

prompt_1 = template.invoke({"product":"智能手机","aspect1":"电池续航","aspect2":"拍照质量"})

print(type(prompt_1))
str = llm.invoke(prompt_1)  #此时的参数prompt_1是PromtValue类型
print(str)

print(type(str))

<class 'langchain_core.prompt_values.StringPromptValue'>
content='智能手机作为现代生活中不可或缺的设备，具有许多优点和缺点。以下是对智能手机的一些评价，特别是关注电池续航和拍照质量。\n\n### 优点：\n\n1. **多功能性**：智能手机集成了电话、短信、社交媒体、浏览器、导航、音乐、视频和游戏等多种功能，满足了用户的多样化需求。\n\n2. **便携性**：智能手机小巧轻便，便于随身携带，使得用户可以随时随地进行通讯和获取信息。\n\n3. **拍照质量**：现代智能手机的摄像头技术不断进步，许多高端机型配备了高分辨率的摄像头和先进的图像处理技术，能够拍摄出色的照片和视频，满足日常使用和社交分享的需求。\n\n4. **应用生态**：丰富的应用程序生态系统使用户能够通过下载应用来扩展手机的功能，涵盖了学习、娱乐、办公等多个领域。\n\n5. **联网能力**：智能手机通过Wi-Fi和移动网络实现了随时随地的互联网访问，方便用户获取信息和进行在线交流。\n\n### 缺点：\n\n1. **电池续航**：尽管智能手机的功能强大，但高性能的处理器和大屏幕往往导致电池消耗较快。特别是在使用高强度应用（如游戏、视频播放和导航）时，电池续航成为一个显著问题，需要频繁充电。\n\n2. **使用习惯**：智能手机的普及使得人们越来越依赖手机，可能导致社交隔离、注意力分散和健康问题（如颈椎病、视力下降等）。\n\n3. **隐私和安全**：智能手机存储了大量个人信息，容易受到黑客攻击、病毒和恶意软件的威胁。同时，用户的隐私可能被应用程序收集和滥用。\n\n4. **价格**：高端智能手机的价格通常较高，普通用户在选择时可能面临经济压力。\n\n5. **拍照局限**：尽管智能手机的拍照能力显著提升，但在低光环境、快速移动物体以及专业摄影需求方面，仍然无法与专业相机相提并论。\n\n### 总结：\n\n智能手机无疑带来了便利和多样化的功能，但在电池续航和使用体验等方面仍然存在一定的局限性。用户在选择和使用智能手机时，需要平衡这些优缺点，以便最大限度地发挥其价值。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'

使用对话大模型进行调用：


In [22]:
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

llm = ChatOpenAI(
	model="gpt-5-mini"
)

In [24]:
template = PromptTemplate.from_template(
    template="请评价{product}的优缺点，包括{aspect1}和{aspect2}。")

prompt_1 = template.invoke({"product":"智能手机","aspect1":"电池续航","aspect2":"拍照质量"})


message = llm.invoke(prompt_1, temperature=1)  #此时的参数prompt_1是PromtValue类型
print(message)

print(type(message))  #<class 'langchain_core.messages.ai.AIMessage'>

content='下面给出智能手机总体的优缺点，并对“电池续航”和“拍照质量”做更详细的说明与实用建议，方便你选购或优化使用。\n\n总体优点\n- 通信与连接：随时通话、短信、社交、视频会议和导航。\n- 多功能：集成摄影、音乐、支付、办公、娱乐和健康追踪，替代多种独立设备。\n- 便携性：体积小、随身携带，随时上网和处理事务。\n- 软件生态：应用商店里的丰富应用满足各种需求。\n- 持续更新：系统与应用不断迭代带来新功能和安全修复。\n\n总体缺点\n- 电池限制：需频繁充电，长时间高负载会不便。\n- 屏幕与隐私：长时间使用易眼疲劳，存在隐私与数据泄露风险。\n- 易碎与维修成本：玻璃机身、密封设计使维修和更换成本较高，部分品牌维护窗口有限。\n- 使用分心：通知与应用容易分散注意力，影响效率与睡眠。\n- 折旧与环保：更新换代快，电子垃圾问题明显。\n\n电池续航（详细）\n- 影响因素：电池容量（常见3000–5000mAh）、处理器能效、屏幕尺寸与刷新率（60Hz vs 90/120Hz）、网络（5G耗电高于4G）、后台应用、定位与同步频率、温度和电池老化。\n- 典型表现：普通手机在中度使用下屏幕点亮时间（SoT）约4–8小时；续航强的机型（日常可用1–2天）通常有大容量电池或更高能效优化。\n- 充电速度与寿命：快充（例如30–120W）可在几十分钟内大幅充电，但长期高功率充电会加速电池化学老化。锂电池一般每年有5–20%的容量衰减，取决于充放电习惯与温度。\n- 优化建议：\n  - 关闭高刷新率或降为60Hz；调低亮度；合理使用省电模式。\n  - 关闭不必要的后台应用、定位、蓝牙与同步。\n  - 避免极端温度；尽量不要长时间把电池维持在0%或100%（常保持20–80%有利寿命）。\n  - 若重视续航，优先看官方标注续航、屏幕尺寸、处理器工艺和系统优化。\n\n拍照质量（详细）\n- 决定因素：传感器尺寸（越大越好）、像素尺寸（单像素越大感光越强）、镜头光学素质与光圈、光学防抖（OIS）、镜头组合（广角、超广、长焦、潜望）、图像处理（ISP和算法/计算摄影）。\n- 计算摄影：现代手机大量依靠软件合成（HDR、夜景、降噪、人像分割等），因此单看“百万像素”并不能代表真实成像质量。\n- 实际表现：\n  - 日间：多数中高端手机拍照优秀，

# 到目前为止的调用的总结：
第1步：提供大语言模型的实例：非对话类的、对话类的

第2步：提供提示词模板的对象：PrompTemplate(template="",input_variables=) / from_template(template)

第3步：将提示词模板对象中的变量进行赋值：format() / invoke() ; partial()

第4步：将填充完变量以后的调用结构传入大模型的invoke(PromptValue / Str)中